In [1]:
!git clone -b robs-branch --single-branch https://github.com/RobertDavidson1/GrokkingExperiments.git
%cd /content/GrokkingExperiments

%pip install --no-deps -e .
%pip install --no-deps scienceplots

fatal: destination path 'GrokkingExperiments' already exists and is not an empty directory.
/content/GrokkingExperiments
Obtaining file:///content/GrokkingExperiments
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for grokkingexperiments (pyproject.toml) ... done
  Created wheel for grokkingexperiments: filename=grokkingexperiments-0.1.0-py3-none-any.whl size=3482 sha256=d28ee81ddde10c96945f4514b0b40cab567511e36922f488a188f0a2babff7e9
  Stored in directory: /tmp/pip-ephem-wheel-cache-ceqk74bn/wheels/d3/e5/64/e6e5467af90300195e6acf9aa85f648301f26229be08833fdd
Successfully built grokkingexperiments
  Attempting uninstall: grokkingexperiments
    Found existing installation: grokkingexperiments 0.1.0
    Uninstalling grokkingexperiments-0.1.0:
      Successfully uninstalled grokkingexperiments-0.1.0
  Using cac

In [2]:
from grokkingexperiments.config import print_config, load_config
from grokkingexperiments.model import GrokkingModel
from grokkingexperiments.groups.zn import Zn

import torch
import numpy as np
import matplotlib.pyplot as plt

import scienceplots
plt.style.use(["science", "ieee", "notebook"])
# plt.style.use(["science", "notebook"])


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} to train")

Using cuda to train


In [3]:
config = load_config()
config.group.n  = 112
config.train.epochs = 250_000
config.progress_interval = 5000

print_config(config)




Config
  model
    | actv_func            relu
    | embed_dim            128
    | hidden_dim           256
  train
    | epochs               250000
  group
    | name                 Zn
    | n                    112
    | split                0.4
  settings
    | seed                 1337
    | print_progress       True
    | progress_interval    100
    | model_dir            data/models/
    | model_filename       custom_model.pt


In [4]:
left_embed_weights = {}
n_seeds = 30

for seed in range(n_seeds):
    print(f"Training model with seed {seed}")
    config.settings.seed = seed # Overwrite seed

    model = GrokkingModel(config).to(device)
    zn = Zn(config)

    # Generate the data
    x_train, y_train, x_test, y_test = zn.generate_train_test_data()

    # Move to GPU if available
    x_train = x_train.to(device)
    y_train = y_train.to(device)
    x_test = x_test.to(device)
    y_test = y_test.to(device)

    # Train the model
    model.fit(x_train, y_train, x_test, y_test)


    E_L = model.state_dict()['E_L']
    left_embed_weights[seed] = E_L.detach().clone().to("cpu")

Trainin model with seed 0
Epoch  0      | Loss 4.721  | Test loss 4.722  | Train accuracy 0.008  | Test accuracy 0.007 
Epoch  100    | Loss 3.481  | Test loss 6.087  | Train accuracy 0.195  | Test accuracy 0.000 
Epoch  200    | Loss 0.996  | Test loss 10.082 | Train accuracy 0.977  | Test accuracy 0.000 
Epoch  300    | Loss 0.163  | Test loss 12.804 | Train accuracy 1.000  | Test accuracy 0.000 
Epoch  400    | Loss 0.090  | Test loss 12.814 | Train accuracy 1.000  | Test accuracy 0.000 
Epoch  500    | Loss 0.067  | Test loss 12.313 | Train accuracy 1.000  | Test accuracy 0.000 
Epoch  600    | Loss 0.054  | Test loss 11.684 | Train accuracy 1.000  | Test accuracy 0.000 
Epoch  700    | Loss 0.045  | Test loss 11.034 | Train accuracy 1.000  | Test accuracy 0.000 
Epoch  800    | Loss 0.040  | Test loss 10.506 | Train accuracy 1.000  | Test accuracy 0.000 
Epoch  900    | Loss 0.035  | Test loss 10.073 | Train accuracy 1.000  | Test accuracy 0.000 
Epoch  1000   | Loss 0.031  | Test

KeyboardInterrupt: 